# Coverage-Risk Curves and Selective Prediction

**Duration:** 25 minutes  
**Level:** Advanced  
**Prerequisites:** Complete [Notebook 02: Calibration Metrics](02_basics_beyond_accuracy_metrics.ipynb)

## Overview

**Coverage-Risk** asks: *Can the system safely abstain from risky predictions?*

In clinical settings, a CDSS should be able to identify uncertain cases and defer to human clinicians. This is **selective prediction** trading coverage (fraction of cases handled) for reduced conditional risk (error rate on handled cases).

### What You'll Learn:

1. Coverage-Risk curves and their interpretation
2. Area Under Risk-Coverage Curve (AURC)
3. Abstention strategies and thresholds
4. Stratified selective prediction by risk tier
5. Multi-model comparison of selective capabilities

### Why This Matters:

**Safety through abstention:**
- High-confidence predictions: CDSS acts autonomously
- Low-confidence predictions: CDSS escalates to human expert
- **Unsafe systems** cannot distinguish these cases

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import BASICS-CDSS coverage-risk module
from basics_cdss.metrics.coverage_risk import (
    coverage_risk_curve,
    area_under_risk_coverage_curve,
    selective_prediction_metrics,
    abstention_rate,
    stratified_selective_metrics
)
from basics_cdss.visualization.coverage_risk_plots import (
    plot_coverage_risk_curve,
    plot_selective_prediction_comparison,
    plot_abstention_analysis
)

print("✓ BASICS-CDSS coverage-risk module loaded")

## 2. Generate Mock CDSS Data

We'll create CDSS predictions with varying confidence levels to demonstrate selective prediction.

In [ ]:
np.random.seed(42)
n_samples = 600

# Generate risk tiers
risk_tiers = np.random.choice(
    ['high', 'medium', 'low'],
    size=n_samples,
    p=[0.25, 0.35, 0.40]
)

# True labels (1 = needs intervention)
y_true = np.zeros(n_samples, dtype=int)
y_true[risk_tiers == 'high'] = np.random.binomial(1, 0.75, (risk_tiers == 'high').sum())
y_true[risk_tiers == 'medium'] = np.random.binomial(1, 0.40, (risk_tiers == 'medium').sum())
y_true[risk_tiers == 'low'] = np.random.binomial(1, 0.15, (risk_tiers == 'low').sum())

# Simulate CDSS predictions with varying confidence
# Model A: Good selective prediction (high confidence correlates with correctness)
base_probs = y_true * 0.75 + (1 - y_true) * 0.25
noise = np.random.normal(0, 0.15, n_samples)
y_prob_good_selective = np.clip(base_probs + noise, 0.01, 0.99)

# Model B: Poor selective prediction (confidence uncorrelated with correctness)
uniform_conf = np.random.uniform(0.3, 0.7, n_samples)
y_prob_poor_selective = np.clip(base_probs * 0.3 + uniform_conf * 0.7, 0.01, 0.99)

print(f"Generated {n_samples} CDSS predictions")
print(f"\nRisk tier distribution:")
print(pd.Series(risk_tiers).value_counts())
print(f"\nIntervention rate: {y_true.mean():.1%}")
print(f"\nAverage confidence:")
print(f"  Good selective model: {y_prob_good_selective.mean():.3f}")
print(f"  Poor selective model: {y_prob_poor_selective.mean():.3f}")

## 3. Coverage-Risk Curves

A coverage-risk curve shows the tradeoff between:
- **Coverage**: Fraction of predictions retained (not abstained)
- **Risk**: Conditional error rate on retained predictions

Good selective prediction: Risk decreases as coverage decreases (abstaining improves accuracy).

In [ ]:
# Compute coverage-risk curves
cov_good, risk_good, thresh_good = coverage_risk_curve(
    y_true, y_prob_good_selective, n_thresholds=100
)

cov_poor, risk_poor, thresh_poor = coverage_risk_curve(
    y_true, y_prob_poor_selective, n_thresholds=100
)

# Plot good selective model
fig1, ax1 = plot_coverage_risk_curve(
    cov_good, risk_good,
    title="Good Selective Prediction: Coverage vs Risk",
    color="#06A77D",
    highlight_points=[(0.8, "Target 80% coverage")]
)
plt.show()

# Plot poor selective model
fig2, ax2 = plot_coverage_risk_curve(
    cov_poor, risk_poor,
    title="Poor Selective Prediction: Coverage vs Risk",
    color="#E63946"
)
plt.show()

print("✓ Coverage-risk curves plotted")
print("\nInterpretation:")
print("  - Good curve: Risk drops sharply as coverage decreases")
print("  - Poor curve: Risk remains high even at low coverage")
print("  - Ideal: Steep negative slope (abstention helps)")

## 4. Area Under Risk-Coverage Curve (AURC)

AURC quantifies selective prediction quality:

$$
\text{AURC} = \int_0^1 \text{Risk}(c) \, dc
$$

**Lower is better**: Ideal system maintains low risk across all coverage levels.

In [ ]:
# Compute AURC for both models
aurc_good = area_under_risk_coverage_curve(cov_good, risk_good)
aurc_poor = area_under_risk_coverage_curve(cov_poor, risk_poor)

print("Area Under Risk-Coverage Curve (AURC)")
print("="*60)
print(f"Good Selective Model: {aurc_good:.4f}")
print(f"Poor Selective Model: {aurc_poor:.4f}")
print(f"\nImprovement: {(aurc_poor - aurc_good):.4f}")
print(f"Relative reduction: {((aurc_poor - aurc_good) / aurc_poor * 100):.1f}%")

print("\nInterpretation:")
print("  Lower AURC = better selective prediction capability")
print("  The good model can safely abstain and improve accuracy")

## 5. Comprehensive Selective Prediction Metrics

Use `selective_prediction_metrics()` for full analysis including AURC and operating points.

In [ ]:
# Compute comprehensive metrics
sp_metrics_good = selective_prediction_metrics(
    y_true, y_prob_good_selective,
    target_coverage=0.8,
    target_risk=0.10
)

sp_metrics_poor = selective_prediction_metrics(
    y_true, y_prob_poor_selective,
    target_coverage=0.8,
    target_risk=0.10
)

print("Comprehensive Selective Prediction Metrics")
print("="*70)

print("\nGOOD SELECTIVE MODEL:")
print(f"  AURC:                      {sp_metrics_good.aurc:.4f}")
print(f"  Risk at 80% coverage:      {sp_metrics_good.risk_at_coverage_threshold:.4f}")
print(f"  Coverage at 10% risk:      {sp_metrics_good.coverage_at_risk_threshold:.4f}")

print("\nPOOR SELECTIVE MODEL:")
print(f"  AURC:                      {sp_metrics_poor.aurc:.4f}")
print(f"  Risk at 80% coverage:      {sp_metrics_poor.risk_at_coverage_threshold:.4f}")
print(f"  Coverage at 10% risk:      {sp_metrics_poor.coverage_at_risk_threshold:.4f}")

print("\nKey Insight:")
print("  Good model achieves lower risk at same coverage")
print("  Or achieves higher coverage at same risk tolerance")

## 6. Abstention Analysis

Analyze how abstention rate affects performance across different confidence thresholds.

In [ ]:
# Analyze abstention behavior
fig, axes = plot_abstention_analysis(
    y_prob_good_selective,
    y_true,
    thresholds=np.linspace(0, 1, 21)
)
plt.show()

print("✓ Abstention analysis plotted")

# Compute abstention rates at key thresholds
print("\nAbstention Rates at Key Thresholds:")
print("="*50)
for tau in [0.5, 0.6, 0.7, 0.8]:
    rate = abstention_rate(y_prob_good_selective, threshold=tau)
    print(f"  Threshold {tau:.1f}: {rate:.1%} abstention")

print("\nClinical Decision:")
print("  Choose threshold based on acceptable abstention rate")
print("  Higher threshold = more abstention = lower risk on retained cases")

## 7. Stratified Selective Prediction by Risk Tier

**Critical**: Selective prediction capability may differ by risk tier.
High-risk patients require especially good selective prediction.

In [ ]:
# Compute stratified selective prediction metrics
stratified_sp = stratified_selective_metrics(
    y_true,
    y_prob_good_selective,
    risk_tiers,
    n_thresholds=100
)

print("Stratified Selective Prediction Analysis")
print("="*70)
print(f"{'Risk Tier':<15} {'AURC':<15} {'Samples':<15}")
print("-"*70)

for tier, metrics in stratified_sp.items():
    n_samples_tier = (risk_tiers == tier).sum()
    print(f"{tier:<15} {metrics.aurc:<15.4f} {n_samples_tier:<15}")

print("\nKey Insight:")
print("  Check if AURC is consistent across risk tiers")
print("  High-risk tier AURC is most critical for safety")

## 8. Multi-Model Selective Prediction Comparison

Compare selective prediction capabilities across multiple CDSS models.

In [ ]:
# Compare models
models_sp = {
    "Good Selective": (sp_metrics_good.coverage_curve, sp_metrics_good.risk_curve),
    "Poor Selective": (sp_metrics_poor.coverage_curve, sp_metrics_poor.risk_curve),
}

fig, ax = plot_selective_prediction_comparison(
    models_sp,
    title="Multi-Model Selective Prediction Comparison"
)
plt.show()

print("✓ Multi-model comparison plotted")

# Rank models by AURC
print("\nModel Ranking by AURC (lower is better):")
print("="*50)
rankings = [
    ("Good Selective", sp_metrics_good.aurc),
    ("Poor Selective", sp_metrics_poor.aurc),
]
rankings.sort(key=lambda x: x[1])

for rank, (name, aurc) in enumerate(rankings, 1):
    print(f"{rank}. {name:<20} AURC = {aurc:.4f}")

## 9. Clinical Deployment Strategies

### Setting Abstention Thresholds:

1. **High-Autonomy**: Low threshold (e.g., 0.6)
   - High coverage (80-90%)
   - CDSS handles most cases
   - Acceptable when selective prediction is excellent

2. **Balanced**: Medium threshold (e.g., 0.7)
   - Moderate coverage (60-80%)
   - Standard deployment
   - Recommended for most clinical settings

3. **Safety-First**: High threshold (e.g., 0.8)
   - Low coverage (40-60%)
   - CDSS only acts when very confident
   - Essential for high-stakes decisions

### Red Flags:

- **Flat coverage-risk curve**: Model cannot improve through abstention
- **High AURC**: Poor selective prediction
- **Tier-specific issues**: Different abstention capability by risk level

## 10. Summary and Key Takeaways

### Core Concepts:

1. **Coverage-Risk Curves**: Visualize abstention tradeoff
2. **AURC**: Quantifies selective prediction quality (lower is better)
3. **Abstention Strategies**: Threshold selection based on safety requirements
4. **Stratified Analysis**: Verify selective capability across risk tiers
5. **Multi-Model Comparison**: Identify systems with best selective prediction

### Key API Functions:

```python
# Coverage-risk analysis
coverages, risks, thresholds = coverage_risk_curve(y_true, y_prob)
aurc = area_under_risk_coverage_curve(coverages, risks)

# Comprehensive metrics
sp_metrics = selective_prediction_metrics(y_true, y_prob)

# Stratified by risk tier
stratified_sp = stratified_selective_metrics(y_true, y_prob, risk_tiers)
```

### Next Steps:

- **[Notebook 04](04_harm_aware_evaluation.ipynb)**: Harm-aware metrics and cost analysis
- **[Notebook 05](05_end_to_end_pipeline.ipynb)**: Complete evaluation workflow

### Clinical Impact:

Selective prediction enables:
- **Safe automation**: CDSS acts only when confident
- **Human-AI collaboration**: Natural escalation to experts
- **Risk mitigation**: Avoid errors through appropriate abstention